# Projet 9 — Petit LLM : fine-tuning QLoRA pour l'extraction d'informations en JSON (variante A)

**Question** : un petit modèle ouvert (Qwen2.5-1.5B-Instruct, 4 bits) adapté par QLoRA extrait-il mieux
les entités (personnes, organisations, lieux) au format JSON que **le même modèle avec prompt seul**
(zero-shot et few-shot), sur un jeu de test jamais vu ?

* Toute la logique est dans le paquet `src/llmx` (testé) ; ce notebook ne fait qu'orchestrer.
* Tous les hyperparamètres sont dans `configs/base.yaml`.
* Le jeu de **test n'est utilisé qu'une fois**, à la fin (verrou `reports/test_locks/`).

## 0. Installation et environnement

In [ ]:
import os, sys, pathlib

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # un seul GPU (Kaggle fournit parfois 2 x T4)
REPO_URL = "https://github.com/Aziz-Gnouma/llm-extract.git"
# Kaggle se reconnaît à sa variable d'environnement (Colab possède aussi un dossier /kaggle)
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
BASE = pathlib.Path("/kaggle/working" if ON_KAGGLE else "/content")
ROOT = BASE / "llm-extract"
if not ROOT.exists():
    !git clone -q {REPO_URL} {ROOT}
%cd {ROOT}
!pip install -q -e ".[ml,dev]"
sys.path.insert(0, str(ROOT / "src"))   # évite un redémarrage du noyau après l'installation editable

In [ ]:
import json, platform, importlib.metadata as md
import torch

env = {"python": platform.python_version(),
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       **{p: md.version(p) for p in ["torch", "transformers", "peft", "bitsandbytes", "accelerate", "datasets"]}}
print(json.dumps(env, indent=2))
assert env["gpu"], "Activez un GPU (T4) dans les paramètres du notebook"
pathlib.Path("reports").mkdir(exist_ok=True)
pathlib.Path("reports/environment.json").write_text(json.dumps(env, indent=2))

In [ ]:
!python -m pytest -q   # tests unitaires : parsing, métriques, découpages, masquage des labels

## 1. Données : CoNLL-2003 → JSON

Chaque phrase devient `{"persons": [...], "organizations": [...], "locations": [...]}` (MISC est écarté).
Découpage **figé** : test tiré du *test* officiel, validation du *validation*, entraînement du *train* ;
doublons supprimés et aucune phrase partagée entre ensembles (vérifié par assertion + hash SHA-256).

In [ ]:
import pandas as pd
from llmx.config import load_config, set_seed
from llmx import pipeline

cfg = load_config("configs/base.yaml")
set_seed(cfg.seed)
manifest = pipeline.prepare_data(cfg)
splits = pipeline.load_splits(cfg)
pd.DataFrame({k: {"n": v["n"], **v["entity_counts"], "sha256": v["sha256"][:12]}
              for k, v in manifest["splits"].items()}).T

In [ ]:
import matplotlib.pyplot as plt

lengths = {k: [len(r["text"].split()) for r in v] for k, v in splits.items()}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
counts = pd.DataFrame({k: v["entity_counts"] for k, v in manifest["splits"].items()}).T
(counts.drop(columns="sentences_without_entity").div(counts.sum(axis=1), axis=0)
       .plot.bar(ax=ax[0], title="Répartition des entités par ensemble", rot=0))
ax[1].hist([lengths["train"], lengths["test"]], bins=30, label=["train", "test"], density=True)
ax[1].set_title("Longueur des phrases (tokens)"); ax[1].legend()
plt.tight_layout(); plt.savefig("reports/data_overview.png", dpi=120)
for r in splits["train"][:3]: print(r["text"], "\n  →", r["entities"])

## 2. Baselines : même modèle, prompt seul (validation)

Le prompt système est **identique** pour toutes les variantes ; le few-shot ajoute k exemples tirés
du *train* uniquement. Baseline triviale à garder en tête : répondre toujours « vide » donne
une exact match égale à la part de phrases sans entité (~21 %) et un F1 de 0.

In [ ]:
zero = pipeline.run_evaluation(cfg, "zero_shot", "val")
k = cfg.baseline.few_shot_k
few = pipeline.run_evaluation(cfg, f"few_shot{k}", "val", k_shots=k)
pd.DataFrame(pipeline.collect_metrics(cfg))

## 3. Fine-tuning QLoRA

In [ ]:
report = pipeline.run_training(cfg)
print({k: v for k, v in report.items() if k != "log_history"})

In [ ]:
log = pd.DataFrame(report["log_history"])
fig, ax = plt.subplots(figsize=(7, 3.5))
log.dropna(subset=["loss"]).plot(x="step", y="loss", ax=ax, label="train loss")
log.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, marker="o", label="val loss")
ax.set_title("Courbes d'apprentissage"); plt.tight_layout()
plt.savefig(f"reports/{cfg.run_name}/learning_curves.png", dpi=120)

In [ ]:
qlora = pipeline.run_evaluation(cfg, "qlora", "val", use_adapter=True)
table = pd.DataFrame(pipeline.collect_metrics(cfg)).query("split == 'val'").set_index("system")
table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
lo = table["micro_f1"] - table["f1_ci95"].map(lambda c: c[0])
hi = table["f1_ci95"].map(lambda c: c[1]) - table["micro_f1"]
ax.bar(table.index, table["micro_f1"], yerr=[lo, hi], capsize=6)
ax.set_ylabel("micro-F1 (IC bootstrap 95 %)"); ax.set_ylim(0, 1); ax.set_title("Validation")
plt.tight_layout(); plt.savefig(f"reports/{cfg.run_name}/f1_val.png", dpi=120)

## 4. Expériences de réglage (validation uniquement)

Chaque variante est une copie de la configuration avec un autre `run_name` : ses artefacts sont séparés.
Choisir 2 expériences dans le budget GPU (≈ 15 min chacune) :
* **quantité de données** : 100 / 500 / 1500 exemples (`train.max_train_examples`) ;
* **rang LoRA** : r = 8 contre 16 ;
* taux d'apprentissage : 1e-4 contre 2e-4.

In [ ]:
from dataclasses import replace

def variant(name, **train_overrides):
    return replace(cfg, run_name=f"{cfg.run_name}-{name}", train=replace(cfg.train, **train_overrides))

ablation = []
for n in (100, 500):
    c = variant(f"n{n}", max_train_examples=n)
    pipeline.run_training(c)
    m = pipeline.run_evaluation(c, "qlora", "val", use_adapter=True)
    ablation.append({"n_train": n, "micro_f1": m["micro"]["f1"], "json_valid": m["json_valid_rate"]})
ablation.append({"n_train": len(splits["train"]), "micro_f1": qlora["micro"]["f1"],
                 "json_valid": qlora["json_valid_rate"]})
pd.DataFrame(ablation)

## 5. Analyse d'erreurs (validation)

Taxonomie : `invalid_json`, `hallucination` (valeur absente du texte), `wrong_field` (bonne entité,
mauvais type), `boundary` (frontière trop longue/courte), `spurious` (pas une entité), `missed`.

In [ ]:
from llmx.data.io import read_json, read_jsonl

errors = pd.DataFrame({s: read_json(cfg.run_reports_dir / f"{s}_val" / "metrics.json")["error_summary"]
                       for s in table.index}).T
errors

In [ ]:
preds = read_jsonl(cfg.run_reports_dir / "qlora_val" / "predictions.jsonl")
wrong = [p for p in preds if p["errors"]]
print(f"{len(wrong)} phrases avec au moins une erreur sur {len(preds)}\n")
for p in wrong[:20]:
    print("TEXTE :", p["text"])
    print("  gold :", p["gold"])
    print("  pred :", p["pred"])
    print("  ->", [(e["category"], e["field"], e["value"]) for e in p["errors"]], "\n")

**Hypothèses à rédiger** (≥ 20 exemples examinés) : ambiguïtés ORG/LOC (équipes nommées par leur ville),
noms tout en majuscules (dépêches sportives), abréviations, frontières (« U.S. », titres), étiquettes
bruitées de CoNLL…

## 6. Évaluation finale sur le test — **une seule fois**

Uniquement les systèmes retenus sur la validation : la meilleure baseline et le modèle QLoRA final.
Une seconde exécution lève `HoldoutAlreadyUsedError`.

`RUN_TEST` reste à `False` pendant tout le développement ; on le passe à `True` **une seule fois**,
quand la configuration finale est figée d'après la validation.

In [ ]:
RUN_TEST = False   # passer à True une seule fois, pour la remise finale

if RUN_TEST:
    best_baseline = "zero_shot" if zero["micro"]["f1"] >= few["micro"]["f1"] else f"few_shot{k}"
    pipeline.run_evaluation(cfg, best_baseline, "test",
                            k_shots=0 if best_baseline == "zero_shot" else k)
    pipeline.run_evaluation(cfg, "qlora", "test", use_adapter=True)
    display(pd.DataFrame(pipeline.collect_metrics(cfg)).query("split == 'test'").round(3))
else:
    print("Test non évalué (RUN_TEST = False) : résultats de développement sur la validation uniquement.")

## 7. Analyse critique (à rédiger, ≥ 1 page)

### Résultats
* Gain du fine-tuning sur la baseline prompt seul (F1 + IC 95 %) : significatif ou non ?
* Coût : temps d'entraînement, latence par exemple, mémoire GPU, taille de l'adaptateur.

### Erreurs et hallucinations
* Catégories dominantes et exemples représentatifs ; le fine-tuning réduit-il les hallucinations ?

### Limites
* Un seul jeu (dépêches Reuters anglaises de 1996) : pas de généralisation à d'autres domaines/langues.
* Étiquettes CoNLL partiellement bruitées ; MISC écarté ; correspondance par chaîne exacte normalisée.
* Une seule graine : variance d'entraînement non mesurée.

### Éthique et risques
* Biais du corpus (géographique, genre des personnes citées).
* Extraction de noms de personnes : enjeux RGPD si appliqué à des données réelles.
* Un modèle qui produit un JSON valide peut quand même halluciner : validation humaine nécessaire.